[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/25_flash_attention.ipynb)

# 🔴 Hard: Flash Attention (Tiled)

Implement **tiled attention with online softmax** — the core idea behind Flash Attention.

### Signature
```python
def flash_attention(Q, K, V, block_size=32) -> Tensor:
    # Q, K, V: (B, S, D)
    # Returns: (B, S, D) — same as standard attention
```

### Key Insight
Instead of materializing the full S×S attention matrix, process in blocks:
1. For each Q-block, iterate over K/V blocks
2. Use **online softmax**: track running `max` and `sum`
3. Rescale accumulator when max changes: `acc *= exp(old_max - new_max)`
4. Final: `output = acc / row_sum`

Must give **identical** results to standard softmax attention.

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.8 MB/s eta 0:00:00


In [2]:
import torch
import math

In [38]:
# ✏️ YOUR IMPLEMENTATION HERE

# NOTE: This implementation fails Autograd due to inplace manipulations
def flash_attention(Q, K, V, block_size=32):
    # Process Q in blocks, iterate K/V blocks with online softmax
    B, N, D = Q.shape
    scale = 1 / math.sqrt(D)
    # Running Output tensor
    O = torch.zeros_like(Q)
    # Running Max tensor
    M = torch.full((B, N, 1), -float('inf'))
    # Running Sum tensor
    L = torch.zeros(B, N)

    for batch in range(B):
        # Iterate over Q blocks
        for q_start in range(0, N, block_size):
            q_end = min(N, q_start + block_size)
            Q_block = Q[batch, q_start:q_end, :]

            for kv_start in range(0, N, block_size):
                kv_end = min(N, kv_start + block_size)
                K_block = K[batch, kv_start:kv_end, :]
                V_block = V[batch, kv_start:kv_end, :]

                S_block = Q_block @ K_block.transpose(-2, -1) * scale

                for qi in range(S_block.shape[0]):
                    block_max = S_block[qi].max().item()

                    # Get existing stats
                    old_max = M[batch, q_start + qi].item()
                    old_l = L[batch, q_start + qi]
                    old_o = O[batch, q_start + qi]

                    if block_max > old_max:
                        correction_factor = math.exp(old_max - block_max)
                        # ✅ Fix: Non-inplace updates
                        old_o = old_o * correction_factor
                        old_l = old_l * correction_factor
                        current_max = block_max
                    else:
                        current_max = old_max

                    block_exp = torch.exp(S_block[qi] - current_max)

                    # ✅ Accumulate into local variables
                    new_l = old_l + torch.sum(block_exp)
                    new_o = old_o + (block_exp @ V_block)

                    L[batch, q_start + qi] = new_l
                    O[batch, q_start + qi] = new_o
                    M[batch, q_start + qi] = current_max

        # Final normalization
        O[batch] = O[batch] / L[batch].unsqueeze(1)

    return O

In [44]:
# ✏️ YOUR IMPLEMENTATION HERE

# NOTE: Safe version of flash attention with batch inputs; safe with autograd
def flash_attention(Q, K, V, block_size=32):
    B, N, D = Q.shape
    scale = 1 / math.sqrt(D)

    # To avoid inplace errors, collect finished rows and combine them at the end.
    batch_results = []

    for b in range(B):
        row_results = []
        for i in range(N):
            # Local "registers" for this specific query row
            # These are NOT part of a global tensor being overwritten
            r_max = torch.tensor(-float('inf'), device=Q.device)
            r_sum = torch.tensor(0.0, device=Q.device)
            r_out = torch.zeros(D, device=Q.device)

            qi = Q[b, i:i+1] # Shape (1, D)

            # Scan through all KV blocks for this one Query
            for kv_start in range(0, N, block_size):
                kv_end = min(N, kv_start + block_size)
                ki = K[b, kv_start:kv_end]
                vi = V[b, kv_start:kv_end]

                # block scores
                attn_scores = (qi @ ki.transpose(-2, -1)) * scale
                block_max = torch.max(attn_scores)

                # Update Stats
                new_max = torch.maximum(r_max, block_max)
                # Correction factor
                correction_factor = torch.exp(r_max - new_max)

                # Accumulate into local variables (Safe for Autograd)
                block_exp = torch.exp(attn_scores - block_max) * torch.exp(block_max - new_max)

                r_out = r_out * correction_factor + (block_exp @ vi).squeeze(0)
                r_sum = r_sum * correction_factor + torch.sum(block_exp)
                r_max = new_max

            # 4. Final normalization for this row
            row_results.append(r_out / r_sum)

        batch_results.append(torch.stack(row_results))

    return torch.stack(batch_results)


In [45]:
# 🧪 Debug
import math
Q, K, V = torch.randn(1, 8, 4), torch.randn(1, 8, 4), torch.randn(1, 8, 4)
out = flash_attention(Q, K, V, block_size=4)
scores = torch.bmm(Q, K.transpose(1,2)) / math.sqrt(4)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
print('Match:', torch.allclose(out, ref, atol=1e-4))

Match: True


In [46]:
# ✅ SUBMIT
from torch_judge import check
check('flash_attention')


🧪 Testing: Flash Attention (Tiled) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Matches standard attention (18.7ms)
  ✅ [2/4] Non-aligned block size (3.5ms)
  ✅ [3/4] Block size invariant (7.0ms)
  ✅ [4/4] Gradient flow (8.9ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (38.1ms total)
  Progress saved. Run status() to see your dashboard.

